# Self-Querying Retrieval: Combining Semantic Search with Structured Metadata Filtering

Retrieval-Augmented Generation (RAG) systems are powerful, but their effectiveness often hinges on the quality and specificity of the retrieved context. While basic vector search excels at finding documents semantically similar to a query, it is inherently blind to structured metadata—the critical details like `genre`, `year`, or `director`. This limitation can lead to overly broad results or difficulty in answering highly constrained questions (e.g., "Show me an action movie from the 2010s directed by Nolan").

This notebook introduces **Self-Querying Retrieval**, a sophisticated technique that elevates RAG beyond simple similarity search. Instead of passing the user query directly to the vector store, we employ a specialized LLM agent (the Self-Query Retriever) whose job is to analyze the incoming natural language question and translate it into two components: 1) a semantic embedding for general context, and 2) structured filters (e.g., `genre="sci-fi"` AND `year > 2000`). By combining these elements, we ensure that retrieval is both semantically relevant *and* strictly constrained by the available metadata.

Mastering self-querying is fundamental for building production-grade RAG pipelines and complex agents using frameworks like LangGraph. In advanced agentic workflows, an LLM must often perform a "planning" step before execution. Self-Querying models this planning process: the agent first decides *what kind* of query it needs (a filter, a keyword search, or just a general semantic search) and then executes that precise plan against the knowledge base. By implementing this pattern, developers can build highly accurate, deterministic retrieval steps that significantly reduce hallucination and improve overall system reliability.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand the limitations of basic vector search:** Recognize when semantic similarity alone is insufficient for precise information retrieval.
*   **Implement Self-Querying Retrieval:** Utilize `SelfQueryRetriever` to automatically translate natural language questions into structured metadata filters and embeddings.
*   **Define Metadata Schema:** Use `AttributeInfo` to explicitly teach the LLM which metadata fields are available and how they should be interpreted for filtering purposes.
*   **Build Advanced RAG Components:** Integrate self-querying as a robust, planning step within complex agentic workflows (e.g., in LangGraph).


### Setup and Imports

This cell imports necessary libraries for setting up a sophisticated Retrieval-Augmented Generation (RAG) pipeline. It brings in components like `Document` for data handling, `Chroma` for the vector store, `OpenAIEmbeddings`/`ChatOpenAI` for embedding and LLM services, and crucially, `SelfQueryRetriever` to enable advanced query understanding based on document metadata.


In [2]:
from dotenv import load_dotenv
# Load environment variables (e.g., API keys) from a .env file
from langchain_core.documents import Document
# Import the Chroma vector store implementation
from langchain_chroma import Chroma
# Import embedding and chat models from OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Import SelfQueryRetriever for advanced query construction based on metadata
from langchain_classic.retrievers import SelfQueryRetriever
# Import schema components used by the retriever (e.g., defining attributes)
from langchain_classic.chains.query_constructor.schema import AttributeInfo
# Import a specific translator for Chroma integration with query construction
from langchain_community.query_constructors.chroma import ChromaTranslator


In [3]:
load_dotenv()

True

### Model Initialization

This cell initializes the core components for our advanced RAG system: the embedding model and the Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the conversational AI engine that processes prompts and generates responses.


In [13]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

llm = ChatOpenAI(model="gpt-5", temperature=0)


### Data Preparation: Loading Structured Documents

This cell initializes a list of `Document` objects, simulating the loading of structured knowledge (movie summaries). Each document contains rich text content and detailed metadata (title, genre, year, etc.), which is crucial for advanced filtering techniques like self-querying.


In [5]:
# Movies dataset — rich, structured metadata makes self-query filtering meaningful
docs = [
    Document(
        page_content="A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An iconic supervillain pushes him to his limits in a battle for Gotham's soul.",
        metadata={"title": "The Dark Knight", "genre": "action", "year": 2008, "rating": 9.0, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A thief who steals secrets through dream-sharing technology is offered a chance to have his past erased if he can plant an idea in someone's mind. A visually stunning exploration of the subconscious.",
        metadata={"title": "Inception", "genre": "sci-fi", "year": 2010, "rating": 8.8, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A team of explorers travels through a wormhole in space to find a new habitable planet for humanity. Stunning visuals of black holes and time dilation challenge our understanding of physics.",
        metadata={"title": "Interstellar", "genre": "sci-fi", "year": 2014, "rating": 8.6, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A programmer discovers that reality is a simulation and joins a rebellion against the machines controlling humanity. A groundbreaking blend of philosophy, martial arts, and bullet-time action.",
        metadata={"title": "The Matrix", "genre": "sci-fi", "year": 1999, "rating": 8.7, "director": "Lana Wachowski"},
    ),
    Document(
        page_content="Two criminals and a mob boss's wife are caught in a web of violence and dark humor over a single eventful day in Los Angeles. Interweaving storylines told out of chronological order.",
        metadata={"title": "Pulp Fiction", "genre": "drama", "year": 1994, "rating": 8.9, "director": "Quentin Tarantino"},
    ),
    Document(
        page_content="A maverick surgeon navigates the chaotic social landscape of a mobile army unit during the Korean War. Sharp satirical comedy disguised as a war film, later adapted into a beloved TV series.",
        metadata={"title": "MASH", "genre": "comedy", "year": 1970, "rating": 7.4, "director": "Robert Altman"},
    ),
    Document(
        page_content="Humanity sends a last-ditch mission to reignite the dying sun with a massive stellar bomb. An intense psychological thriller set in the terrifying emptiness of deep space.",
        metadata={"title": "Sunshine", "genre": "sci-fi", "year": 2007, "rating": 7.3, "director": "Danny Boyle"},
    ),
    Document(
        page_content="A soldier wakes up in another man's body aboard a commuter train just minutes before it explodes, reliving the event repeatedly to identify the bomber. A clever sci-fi thriller about time loops and identity.",
        metadata={"title": "Source Code", "genre": "sci-fi", "year": 2011, "rating": 7.5, "director": "Duncan Jones"},
    ),
]

# Print the total count of documents loaded into the list
print(f"Created {len(docs)} movie documents")


Created 8 movie documents


### Vector Store Initialization (Chroma)

This cell initializes a persistent vector store using ChromaDB. It takes the loaded documents (`docs`) and generates embeddings for them using the specified `embeddings` model, storing these vectors in a collection named `movies_collection`. This makes the document content searchable by semantic similarity.


In [6]:
# Chroma stores embeddings persistently in memory for this session
vectorstore = Chroma.from_documents(docs, embedding=embeddings,
                                    collection_name="movies_collection")


### Metadata Schema Definition

This cell defines the metadata schema using `AttributeInfo`. This structure is crucial for advanced RAG systems (like those utilizing LangGraph) because it explicitly tells the LLM which fields are available in the retrieved documents, what they represent, and what data type to expect. This allows the system to generate precise filtering queries (e.g., 'year > 2020' or 'genre = action') before calling the vector store.


In [7]:
# AttributeInfo tells the LLM what metadata fields exist and how to filter on them

metadata_field_info = [
    AttributeInfo(name="title", description="The title of the movie", type="string"),
    AttributeInfo(name="genre", description="The genre of the movie (action, sci-fi, drama, comedy)", type="string"),
    AttributeInfo(name="year", description="The year the movie was released", type="integer"),
    AttributeInfo(name="rating", description="The IMDb rating of the movie (0-10)", type="float"),
    AttributeInfo(name="director", description="The director of the movie", type="string"),
]



This cell initializes a string variable, `document_content_description`, which holds the specific domain knowledge or context (in this case, movie plot descriptions) that the RAG system will be designed to process and query. This variable acts as a placeholder for the source material used during indexing and retrieval.


In [8]:
document_content_description = "Brief plot descriptions of movies" # Defines the domain-specific content description string that will serve as the knowledge base context.


### Self-Querying Retriever Initialization

This cell initializes a `SelfQueryRetriever`, which is an advanced component that automatically translates natural language questions into optimal search queries (including metadata filters) before querying the vector store. It uses the provided LLM and structured translator to enhance retrieval accuracy.


In [19]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    enable_limit=True
)


### Code Explanation

This cell initializes a retriever object from the `vectorstore` and uses it to query the stored documents. It retrieves the top 3 most similar documents (k=3) based on the input query, simulating an advanced retrieval step crucial for grounding LLM responses.


In [17]:
# create the vs retriever
# Initialize a retriever object from the vectorstore.
# We use 'similarity' search type and specify that we want the top 3 results (k=3).
vs_retriever = vectorstore.as_retriever(search_type="similarity",
                                        search_kwargs={"k": 3})

# Invoke the retriever with a specific query string.
# This executes the search and returns a list of relevant document objects (results).
results = vs_retriever.invoke("What are some sci-fi movies released after 2010")

# Iterate through the retrieved documents to display key information.
for doc in results:
    # Print metadata details (Year, Title, Genre, Director) for structured output.
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']}) - dir. {doc.metadata['director']}")
    # Print the beginning of the document content (page_content) to show context.
    print(f"  {doc.page_content[:100]}...")
    print()


[2007] Sunshine (sci-fi) - dir. Danny Boyle
  Humanity sends a last-ditch mission to reignite the dying sun with a massive stellar bomb. An intens...

[2014] Interstellar (sci-fi) - dir. Christopher Nolan
  A team of explorers travels through a wormhole in space to find a new habitable planet for humanity....

[1999] The Matrix (sci-fi) - dir. Lana Wachowski
  A programmer discovers that reality is a simulation and joins a rebellion against the machines contr...



### Code Explanation

This cell demonstrates the retrieval process using a specialized retriever. It takes a natural language query and uses it to fetch relevant documents from the vector store, implicitly applying metadata filtering (e.g., genre and year) based on the initial LLM extraction step. The loop then iterates through these retrieved documents, printing key metadata fields and a snippet of the document content for review.


In [22]:
# LLM extracts: semantic query="sci-fi movies", filter={genre: "sci-fi", year > 2005}

# Use the retriever to fetch documents based on the user's natural language query.
# The retriever handles both semantic search and metadata filtering (as shown in the comment).
results = retriever.invoke("Recommend me 2 sci-fi movies released after 2000") # metadata --> {"genre": "sci-fi", "year": >= 2005}

# Iterate through the retrieved documents and print their details.
for doc in results:
    # Print key metadata fields (Year, Title, Genre, Director)
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']}) - dir. {doc.metadata['director']}")
    # Print the first 100 characters of the document content snippet
    print(f"  {doc.page_content[:100]}..." )
    print()



[2010] Inception (sci-fi) - dir. Christopher Nolan
  A thief who steals secrets through dream-sharing technology is offered a chance to have his past era...

[2014] Interstellar (sci-fi) - dir. Christopher Nolan
  A team of explorers travels through a wormhole in space to find a new habitable planet for humanity....



### Code Explanation

This cell demonstrates the core retrieval step of RAG. It uses the `retriever` object (which is typically an instance of a vector store query engine) to fetch relevant documents based on a natural language query, effectively simulating how the LLM's semantic understanding guides the search.


In [25]:
# No metadata filter here — LLM uses only the semantic query about dreams/subconscious

# Use the retriever object to invoke a search. The input string is the user's query.
results = retriever.invoke("movie about a superhero who is a billionaire by day and a masked vigilante by night")

# Iterate through the retrieved documents (Document objects)
for doc in results:
    # Print metadata (year, title, genre) for easy identification
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']})")
    # Print a snippet of the document content to show relevance
    print(f"  {doc.page_content[:100]}..." )
    print()


[2008] The Dark Knight (action)
  A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An ic...

[1999] The Matrix (sci-fi)
  A programmer discovers that reality is a simulation and joins a rebellion against the machines contr...

[1994] Pulp Fiction (drama)
  Two criminals and a mob boss's wife are caught in a web of violence and dark humor over a single eve...

[2010] Inception (sci-fi)
  A thief who steals secrets through dream-sharing technology is offered a chance to have his past era...



### Code Explanation

This cell executes the retrieval step using a pre-configured `retriever`. It takes the user's query (which is implicitly refined by the LLM extraction) and fetches relevant documents from the vector store. The subsequent loop iterates through these retrieved documents, printing key metadata (like year, title, and rating) and a snippet of the document content for immediate inspection.


In [26]:
# LLM extracts: semantic query="movies", filter={director: "Christopher Nolan"}

# Use the retriever to fetch relevant documents based on the refined query.
results = retriever.invoke("What movies did Christopher Nolan direct?")

# Iterate through the retrieved documents (Document objects).
for doc in results:
    # Print key metadata fields for easy viewing.
    print(f"[{doc.metadata['year']}] {doc.metadata['title']} - Rating: {doc.metadata['rating']}")
    # Print a snippet of the document content (first 100 characters).
    print(f"  {doc.page_content[:100]}...")
    print()


[2010] Inception - Rating: 8.8
  A thief who steals secrets through dream-sharing technology is offered a chance to have his past era...

[2014] Interstellar - Rating: 8.6
  A team of explorers travels through a wormhole in space to find a new habitable planet for humanity....

[2008] The Dark Knight - Rating: 9.0
  A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An ic...

